# 10. Оценка экономической типологии

Здесь оцениваем уже полученную устойчивую типологию по нескольким направлениям: разделимость типов в пространстве признаков, устойчивость к seed и чувствительность к способу построения экономической близости.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    silhouette_score,
    silhouette_samples,
    calinski_harabasz_score,
    adjusted_rand_score
)
from sklearn.cluster import KMeans

## 1. Загрузка панели

Берем исходные экономические признаки и сохраненную типологию.

In [2]:
features = pd.read_parquet(
    "../data/processed/features_basic.parquet"
)

municipality_types = pd.read_parquet(
    "../data/processed/municipality_economic_types.parquet"
)

cluster_profiles = pd.read_parquet(
    "../data/processed/cluster_profiles.parquet"
)

economic_type_profiles = pd.read_csv(
    "../data/processed/economic_type_profiles.csv",
    index_col=0
)

monthly_type_shares = pd.read_csv(
    "../data/processed/monthly_economic_type_shares.csv"
)

features["date"] = pd.to_datetime(features["date"])
municipality_types["date"] = pd.to_datetime(
    municipality_types["date"]
)
cluster_profiles["date"] = pd.to_datetime(
    cluster_profiles["date"]
)
monthly_type_shares["date"] = pd.to_datetime(
    monthly_type_shares["date"]
)

print("features:", features.shape)
print("municipality_types:", municipality_types.shape)
print("cluster_profiles:", cluster_profiles.shape)
print("economic_type_profiles:", economic_type_profiles.shape)
print("monthly_type_shares:", monthly_type_shares.shape)

features: (50521, 14)
municipality_types: (50521, 4)
cluster_profiles: (521, 8)
economic_type_profiles: (3, 5)
monthly_type_shares: (72, 4)


## 2. Проверка состава типов

Убеждаемся, что итоговая классификация действительно содержит три экономических типа.

In [3]:
print(
    "Экономических типов:",
    municipality_types["economic_type"].nunique()
)

print(
    "Типы:",
    sorted(municipality_types["economic_type"].unique())
)

print(
    "Пропусков:",
    municipality_types["economic_type"].isna().sum()
)

Экономических типов: 3
Типы: [np.int32(0), np.int32(1), np.int32(2)]
Пропусков: 0


## 3. Признаки для оценки

Используем те же пять долей, по которым интерпретируются экономические различия.

In [4]:
model_features = [
    "food_share",
    "health_share",
    "marketplaces_share",
    "catering_share",
    "transport_share"
]

evaluation_data = municipality_types.merge(
    features[
        ["territory_id", "date"] + model_features
    ],
    on=["territory_id", "date"],
    how="left"
)

X_eval = evaluation_data[
    model_features
].to_numpy()

y_eval = evaluation_data[
    "economic_type"
].to_numpy()

print("X:", X_eval.shape)
print("y:", y_eval.shape)

X: (50521, 5)
y: (50521,)


## 4. Silhouette финальной типологии

Оцениваем, насколько четко муниципалитеты отделяются от соседних экономических типов.

In [5]:
silhouette_final = silhouette_score(
    X_eval,
    y_eval,
    metric="euclidean"
)

print(
    "Итоговый Silhouette:",
    silhouette_final
)

Итоговый Silhouette: 0.34893142931275045


## 5. Calinski-Harabasz

Второй атрибутивный индекс проверяет соотношение внутрикластерной и межкластерной вариативности.

In [6]:
calinski_final = calinski_harabasz_score(
    X_eval,
    y_eval
)

print(
    "Итоговый Calinski-Harabasz:",
    calinski_final
)

Итоговый Calinski-Harabasz: 38856.07332914181


## 6. Распределение индивидуальных Silhouette

Смотрим не только среднее значение, но и распределение локального качества по наблюдениям.

In [7]:
silhouette_values = silhouette_samples(
    X_eval,
    y_eval,
    metric="euclidean"
)

evaluation_data["silhouette"] = silhouette_values

type_silhouette = (
    evaluation_data
    .groupby("economic_type")["silhouette"]
    .agg(
        mean_silhouette="mean",
        median_silhouette="median",
        min_silhouette="min"
    )
    .round(4)
)

type_silhouette

,mean_silhouette,median_silhouette,min_silhouette
economic_type,,,
0,0.3450,0.3942,-0.5087
1,0.3535,0.4363,-0.4938
2,0.3508,0.4133,-0.4792


## 7. Профили типов

Сверяем средние экономические доли для каждого устойчивого типа.

In [8]:
X_types = cluster_profiles[
    model_features
].to_numpy()

cluster_sizes = cluster_profiles[
    "n_municipalities"
].to_numpy()

seeds = [1, 42, 100, 2024, 999]

all_type_labels = []

for seed in seeds:

    model = KMeans(
        n_clusters=3,
        random_state=seed,
        n_init=20
    )

    labels_seed = model.fit_predict(
        X_types,
        sample_weight=cluster_sizes
    )

    all_type_labels.append(labels_seed)

print("Запусков:", len(all_type_labels))

Запусков: 5


## 8. Устойчивость к seed

Повторяем KMeans с разными seed и сравниваем разбиения через ARI.

In [9]:
seed_stability = []

for i in range(len(seeds)):

    for j in range(i + 1, len(seeds)):

        ari = adjusted_rand_score(
            all_type_labels[i],
            all_type_labels[j]
        )

        seed_stability.append({
            "seed_1": seeds[i],
            "seed_2": seeds[j],
            "ARI": ari
        })

seed_stability_df = pd.DataFrame(
    seed_stability
)

seed_stability_df

,seed_1,seed_2,ARI
0,1,42,1.0
1,1,100,1.0
2,1,2024,1.0
3,1,999,1.0
4,42,100,1.0
5,42,2024,1.0
6,42,999,1.0
7,100,2024,1.0
8,100,999,1.0
9,2024,999,1.0


## 9. Сводка устойчивости

Агрегируем ARI между повторными запусками.

In [10]:
print(
    "Средний ARI:",
    seed_stability_df["ARI"].mean()
)

print(
    "Минимальный ARI:",
    seed_stability_df["ARI"].min()
)

print(
    "Максимальный ARI:",
    seed_stability_df["ARI"].max()
)

Средний ARI: 1.0
Минимальный ARI: 1.0
Максимальный ARI: 1.0


## 10. Контрольная сеть декабря 2024

Возвращаемся к сетевой постановке и проверяем чувствительность к способу измерения расстояния.

In [11]:
december = (
    features[
        features["date"] == pd.Timestamp("2024-12-01")
    ]
    .copy()
)

X_dec = december[model_features].to_numpy()
municipalities_dec = december["territory_id"].to_numpy()

print("Муниципалитетов:", len(december))

Муниципалитетов: 2103


## 11. Cosine distance

Строим альтернативную cosine-based сеть для sensitivity analysis.

In [12]:
from sklearn.neighbors import NearestNeighbors

k = 10

knn_cosine = NearestNeighbors(
    n_neighbors=k + 1,
    metric="cosine"
)

knn_cosine.fit(X_dec)

distances_cos, indices_cos = knn_cosine.kneighbors(
    X_dec
)

neighbor_distances_cos = distances_cos[:, 1:]
neighbor_indices_cos = indices_cos[:, 1:]

print("Форма:", distances_cos.shape)
print(
    "Медианное cosine distance:",
    np.median(neighbor_distances_cos)
)

Форма: (2103, 11)
Медианное cosine distance: 0.0001293461780194627


In [13]:
sigma_cos = np.median(
    neighbor_distances_cos
)

weights_cos = np.exp(
    -neighbor_distances_cos / sigma_cos
)

print(
    pd.Series(
        weights_cos.ravel()
    ).describe()
)

count    2.103000e+04
mean     3.707420e-01
std      2.469270e-01
min      2.375687e-62
25%      1.616345e-01
50%      3.678794e-01
75%      5.585515e-01
max      9.893328e-01
dtype: float64


## 12. Cosine-граф

Формируем взвешенный неориентированный граф на тех же муниципалитетах.

In [14]:
import networkx as nx

edges_cos = []

for i in range(len(municipalities_dec)):
    for j in range(k):

        neighbor_idx = neighbor_indices_cos[i, j]

        edges_cos.append({
            "source": int(municipalities_dec[i]),
            "target": int(municipalities_dec[neighbor_idx]),
            "weight": float(weights_cos[i, j])
        })

edges_cos = pd.DataFrame(edges_cos)

edges_cos["node_min"] = (
    edges_cos[
        ["source", "target"]
    ].min(axis=1)
)

edges_cos["node_max"] = (
    edges_cos[
        ["source", "target"]
    ].max(axis=1)
)

edges_cos_undirected = (
    edges_cos
    .groupby(
        ["node_min", "node_max"],
        as_index=False
    )["weight"]
    .mean()
    .rename(
        columns={
            "node_min": "source",
            "node_max": "target"
        }
    )
)

In [15]:
G_cos = nx.Graph()

G_cos.add_nodes_from(
    municipalities_dec
)

for row in edges_cos_undirected.itertuples(
    index=False
):
    G_cos.add_edge(
        int(row.source),
        int(row.target),
        weight=float(row.weight)
    )

communities_cos = nx.community.louvain_communities(
    G_cos,
    weight="weight",
    resolution=1.5,
    seed=42
)

labels_cos = np.full(
    len(municipalities_dec),
    -1,
    dtype=int
)

territory_to_position_cos = {
    territory_id: i
    for i, territory_id
    in enumerate(municipalities_dec)
}

for cluster_id, community in enumerate(
    communities_cos
):
    for territory_id in community:
        labels_cos[
            territory_to_position_cos[
                territory_id
            ]
        ] = cluster_id

print(
    "Cosine-кластеров:",
    len(communities_cos)
)

Cosine-кластеров: 33


## 13. Оценка cosine-кластеризации

Сравниваем сетевую и атрибутивную структуру решения, полученного на cosine network.

In [16]:
silhouette_cos = silhouette_score(
    X_dec,
    labels_cos,
    metric="euclidean"
)

calinski_cos = calinski_harabasz_score(
    X_dec,
    labels_cos
)

modularity_cos = nx.community.modularity(
    G_cos,
    communities_cos,
    weight="weight"
)

print("Cosine Silhouette:", silhouette_cos)
print("Cosine Calinski-Harabasz:", calinski_cos)
print("Cosine Modularity:", modularity_cos)

Cosine Silhouette: 0.13617628369822207
Cosine Calinski-Harabasz: 684.2335696456754
Cosine Modularity: 0.8087887720080444


## 14. Euclidean-контроль

Загружаем результаты основной Euclidean-сети для прямого сравнения.

In [17]:
euclidean_edges = pd.read_parquet(
    "../data/processed/edges_2024_12.parquet"
)

euclidean_features = pd.read_parquet(
    "../data/processed/features_2024_12.parquet"
)

euclidean_features["territory_id"].nunique()

2103

In [18]:
G_euclidean = nx.Graph()

euclidean_nodes = (
    euclidean_features["territory_id"]
    .to_numpy()
)

G_euclidean.add_nodes_from(
    euclidean_nodes
)

for row in euclidean_edges.itertuples(index=False):
    G_euclidean.add_edge(
        int(row.source),
        int(row.target),
        weight=float(row.weight)
    )

print("Узлов:", G_euclidean.number_of_nodes())
print("Рёбер:", G_euclidean.number_of_edges())

Узлов: 2103
Рёбер: 13757


## 15. Louvain на Euclidean-сети

Строим контрольное разбиение декабря 2024 на основной мере расстояния.

In [19]:
communities_euclidean = nx.community.louvain_communities(
    G_euclidean,
    weight="weight",
    resolution=1.5,
    seed=42
)

labels_euclidean = np.full(
    len(euclidean_features),
    -1,
    dtype=int
)

territory_to_position_euclidean = {
    territory_id: i
    for i, territory_id
    in enumerate(euclidean_nodes)
}

for cluster_id, community in enumerate(
    communities_euclidean
):
    for territory_id in community:
        labels_euclidean[
            territory_to_position_euclidean[
                territory_id
            ]
        ] = cluster_id

print(
    "Euclidean-кластеров:",
    len(communities_euclidean)
)

Euclidean-кластеров: 19


In [20]:
X_euclidean = euclidean_features[
    model_features
].to_numpy()

silhouette_euclidean = silhouette_score(
    X_euclidean,
    labels_euclidean,
    metric="euclidean"
)

calinski_euclidean = calinski_harabasz_score(
    X_euclidean,
    labels_euclidean
)

modularity_euclidean = nx.community.modularity(
    G_euclidean,
    communities_euclidean,
    weight="weight"
)

print("Silhouette:", silhouette_euclidean)
print("Calinski-Harabasz:", calinski_euclidean)
print("Modularity:", modularity_euclidean)

Silhouette: 0.17842438742691852
Calinski-Harabasz: 914.5988331398605
Modularity: 0.8139722519099776


In [21]:
import pandas as pd

## 16. Сводное сравнение методов

Собираем Euclidean и Cosine в единую таблицу для sensitivity analysis.

In [22]:
comparison_similarity = pd.DataFrame({
    "method": [
        "Euclidean",
        "Cosine"
    ],
    "n_clusters": [
        len(communities_euclidean),
        len(communities_cos)
    ],
    "modularity": [
        modularity_euclidean,
        modularity_cos
    ],
    "silhouette": [
        silhouette_euclidean,
        silhouette_cos
    ],
    "calinski_harabasz": [
        calinski_euclidean,
        calinski_cos
    ]
})

comparison_similarity

,method,n_clusters,modularity,silhouette,calinski_harabasz
0,Euclidean,19,0.813972,0.178424,914.598833
1,Cosine,33,0.808789,0.136176,684.233570


## 17. Сохранение итоговой оценки

Сохраняем показатели, которые затем используются в README и презентации.

In [23]:
evaluation_summary = pd.DataFrame({
    "metric": [
        "final_silhouette",
        "final_calinski_harabasz",
        "seed_ari_mean",
        "seed_ari_min",
        "seed_ari_max"
    ],
    "value": [
        silhouette_final,
        calinski_final,
        seed_stability_df["ARI"].mean(),
        seed_stability_df["ARI"].min(),
        seed_stability_df["ARI"].max()
    ]
})

evaluation_summary.to_csv(
    "../data/processed/evaluation_summary.csv",
    index=False
)

comparison_similarity.to_csv(
    "../data/processed/similarity_method_comparison.csv",
    index=False
)

print("Финальные результаты оценки сохранены.")

Финальные результаты оценки сохранены.
